In [1]:
import os

import matplotlib.pyplot as plt
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from tifffile import imread




In [9]:
path = r"\\storage3.ad.scilifelab.se\testalab\Guillaume\01_Projects\DL_monalisa\_paper\Denoising_Application\Stack-2.tif"
stack = imread(path)
pred = stack[0]
inp = stack[1]
gt = stack[2]

TiffPage 0: TypeError: read_bytes() missing 3 required positional arguments: 'dtype', 'count', and 'offsetsize'


In [10]:
psnr_inp = psnr(gt, inp, data_range=gt.max() - gt.min())
psnr_pred = psnr(gt, pred, data_range=gt.max() - gt.min())
ssim_inp = ssim(gt, inp, data_range=gt.max() - gt.min())
ssim_pred = ssim(gt, pred, data_range=gt.max() - gt.min())

In [11]:
print("Inp PSNR: ", psnr_inp)
print("Pred PSNR: ", psnr_pred)
print("Inp SSIM: ", ssim_inp)
print("Pred SSIM: ", ssim_pred)

Inp PSNR:  25.75197579748169
Pred PSNR:  31.75676633959086
Inp SSIM:  0.5107620115176184
Pred SSIM:  0.8612729940081335


In [4]:
# Function to do box plot

def box_plot(filling_factors,snr_values,values,offset,box_width,colors,metric_name,ax):

    for idx, ff in enumerate(filling_factors):

        # Apply offset to the x positions for each filling factor
        displaced_x = np.array(snr_values) + (idx - 1) * offset
        
        # box plot
        data = [values[snr][ff] for snr in snr_values]
        box = ax.boxplot(data, positions=displaced_x, widths=box_width,showfliers=False)
        
        # Color each box based on the SNR values
        # for i, patch in enumerate(box['boxes']):
        #     plt.setp(patch, facecolor=colors[i])  # Color each box based on SNR

        # Optional: Customize whiskers, caps, medians, and fliers to have the same color
        for item in ['whiskers', 'caps', 'fliers', 'medians','boxes']:
            plt.setp(box[item], color=colors[idx])  # Color the other parts of the boxplot

        # mean and std plot
        # axs[1].errorbar(displaced_x, [mean_ssim[snr][ff] for snr in snr_values], 
        #                 yerr=[std_ssim[snr][ff] for snr in snr_values], 
        #                 label=f'Filling Factor {ff}', fmt='o', capsize=5)


        ax.set_title(metric_name)
        ax.set_xlabel('SNR')
        ax.set_ylabel(metric_name)
        ax.set_xticks(snr_values)

    legend_lines = [Line2D([0], [0], color=colors[idx], lw=1) for idx in range(len(filling_factors))]
    ax.legend(legend_lines, [f'Filling Factor {ff}' for ff in filling_factors], loc='upper right')


In [5]:
use_windowed = True
window_size = 64


# Folder containing the image stacks
folder_path = r'E:\dl_monalisa\Models\Vim_fixed_mltplSNR_30nm\Upsampling_selected\unpaired'

# SNR and filling factor values
snr_values = [1, 2, 3]  # Update according to your SNR values
filling_factors = [0.25, 0.5, 0.75]

# Initialize dictionaries to store metrics for each SNR and filling factor combination
psnr_values = {snr: {ff: [] for ff in filling_factors} for snr in snr_values}
ssim_values = {snr: {ff: [] for ff in filling_factors} for snr in snr_values}

# Process each image stack
for file_name in os.listdir(folder_path):
    if file_name.endswith('.tif'):
        file_path = os.path.join(folder_path, file_name)
        img_stack = imread(file_path)

        # Ground truth (first image in the stack)
        gt = img_stack[0]
        data_range = np.max(gt) - np.min(gt)
        # Process predictions for each SNR and filling factor
        # Process predictions for each SNR and filling factor
        for i, snr in enumerate(snr_values):
            for j, filling_factor in enumerate(filling_factors):
                # Get the corresponding prediction index
                pred_index = i * len(filling_factors) + j + 1
                pred = img_stack[pred_index]
                
                # Calculate PSNR and SSIM
                psnr_val, ssim_val = calculate_metrics(gt, pred, data_range,use_windowed,window_size)
                psnr_values[snr][filling_factor].extend(psnr_val)
                ssim_values[snr][filling_factor].extend(ssim_val)

# Calculate mean and std for each SNR and filling factor combination
mean_psnr = {snr: {ff: np.mean(psnr_values[snr][ff]) for ff in filling_factors} for snr in snr_values}
std_psnr = {snr: {ff: np.std(psnr_values[snr][ff]) for ff in filling_factors} for snr in snr_values}
mean_ssim = {snr: {ff: np.mean(ssim_values[snr][ff]) for ff in filling_factors} for snr in snr_values}
std_ssim = {snr: {ff: np.std(ssim_values[snr][ff]) for ff in filling_factors} for snr in snr_values}


In [ ]:
psnr_values[1][0.5]

In [ ]:
offset = 0.15
box_width = 0.08
colors = ['tab:blue', 'tab:red','tab:green']
fig, axs = plt.subplots(1, 2, figsize=(14, 6))

box_plot(filling_factors,snr_values,psnr_values,offset,box_width,colors,"PSNR",axs[0])
box_plot(filling_factors,snr_values,ssim_values,offset,box_width,colors,"SSIM",axs[1])

In [ ]:
snr_values

In [ ]:
# Plot the metrics with SNR on the x-axis and filling factors as different colors

box_width = 0.08
fig, axs = plt.subplots(1, 2, figsize=(14, 6))
colors = ['tab:blue', 'tab:red','tab:green']

# Horizontal displacement factor (a small offset for each filling factor)
offset = 0.15  # Adjust this value to control the displacement

# Plot PSNR
for idx, ff in enumerate(filling_factors):
    # Apply offset to the x positions for each filling factor
    displaced_x = np.array(snr_values) + (idx - 1) * offset
    
    # box plot
    data_psnr = [psnr_values[snr][ff] for snr in snr_values]
    axs[0].boxplot(data_psnr, positions=displaced_x, widths=box_width,showfliers=False)

    # mean and std plot
    # axs[0].errorbar(displaced_x, [mean_psnr[snr][ff] for snr in snr_values], 
    #                 yerr=[std_psnr[snr][ff] for snr in snr_values], 
    #                 label=f'Filling Factor {ff}', fmt='o', capsize=5)

axs[0].set_title('PSNR')
axs[0].set_xlabel('SNR')
axs[0].set_ylabel('PSNR (dB)')
axs[0].legend()
axs[0].set_xticks(snr_values)


# Plot SSIM
for idx, ff in enumerate(filling_factors):
    # Apply offset to the x positions for each filling factor
    displaced_x = np.array(snr_values) + (idx - 1) * offset
    
    # box plot
    data_ssim = [ssim_values[snr][ff] for snr in snr_values]
    box = axs[1].boxplot(data_ssim, positions=displaced_x, widths=box_width,showfliers=False)
    
    # Color each box based on the SNR values
    # for i, patch in enumerate(box['boxes']):
    #     plt.setp(patch, facecolor=colors[i])  # Color each box based on SNR

    # Optional: Customize whiskers, caps, medians, and fliers to have the same color
    for item in ['whiskers', 'caps', 'fliers', 'medians','boxes']:
        plt.setp(box[item], color=colors[idx])  # Color the other parts of the boxplot

    # mean and std plot
    # axs[1].errorbar(displaced_x, [mean_ssim[snr][ff] for snr in snr_values], 
    #                 yerr=[std_ssim[snr][ff] for snr in snr_values], 
    #                 label=f'Filling Factor {ff}', fmt='o', capsize=5)


axs[1].set_title('SSIM')
axs[1].set_xlabel('SNR')
axs[1].set_ylabel('SSIM')
axs[1].legend()

axs[1].set_xticks(snr_values)
axs[1].legend([f'Filling Factor {ff}' for ff in filling_factors])
plt.tight_layout()
plt.show()
